# Pytorch batched training


Explore some more features on the (very small) Diabetes dataset: Mainly batched training, 

In [ ]:
# complete imports if needed
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split

import torch
import torch.nn as nn
import torch.optim as optim



In [ ]:
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

In [ ]:
from sklearn.datasets import load_diabetes

data = load_diabetes()

X = pd.DataFrame(data.data, columns=data.feature_names)
y = pd.DataFrame(data.target)

print(y.shape)
print(X.head())

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

Use the dataloader to split the dataset into smaller batches (have a look how different batch sizes compare). Hint: Don't go for too small batch sizes, it will massively slow down the training!

In [ ]:
from torch.utils.data import Dataset, DataLoader

class MyDataFromDF(Dataset):
    def __init__(self, X, y):
        # features provided as Dataframes converted
        self.X = torch.tensor(    
            X.to_numpy(), 
            dtype=torch.float32 # optionally include a datatype
            ) 
        
        # targets provided as Dataframes
        self.y = torch.tensor( 
            y.to_numpy(),
            dtype=torch.float32 # optionally include a datatype
            )            

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]   # always return (features, target)

In [ ]:
train_dataset = MyDataFromDF(X_train, y_train)
test_dataset  = MyDataFromDF(X_test, y_test)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader  = DataLoader(test_dataset, batch_size=32, shuffle=False)

Define the NN according to your first attempts.

In [ ]:
class DiabetesNN(nn.Module):
    def __init__(self, input_size, hidden_size, output_size, drop_out):
        super(DiabetesNN, self).__init__()
        self.fc1 = nn.Linear(input_size, hidden_size)  # input layer with width = length of the feature set
        self.dropout1 = nn.Dropout(p=drop_out) # only parts of the neurons are used.

        self.fc2 = nn.Linear(hidden_size, hidden_size)  # one hidden layer, try to add another one
        self.dropout2 = nn.Dropout(p=drop_out)

        self.fc3 = nn.Linear(hidden_size, int(hidden_size/2))  # one hidden layer, try to add another one
        self.dropout3 = nn.Dropout(p=drop_out)

        self.fc4 = nn.Linear(int(hidden_size/2), output_size)  # output layer
    
    # Good practice: activation functions specified in forward pass - separated from the learning parts
    def forward(self, x):            # forward pass
        x = torch.relu(self.fc1(x))  # ReLU activation function for input layer
        x = self.dropout1(x)

        x = torch.relu(self.fc2(x))  # ReLU activation function for hidden layer
        x = self.dropout2(x)

        x = torch.relu(self.fc3(x))  # ReLU activation function for hidden layer
        x = self.dropout3(x)
        
        x = self.fc4(x)   # no activation function for the output layer (because it is a regresion model!)
        return x

In [ ]:
# Hyperparameters
input_size = X_train.shape[1]
hidden_size = 100
output_size = 1
drop_out=0.2 # specifies how much of the NN remains randomly inactive

learning_rate = 0.005

# Number of training iterations
num_epochs = 300

In [ ]:
import torch.optim as optim
model = DiabetesNN(input_size, hidden_size, output_size, drop_out).to(device)

criterion = nn.MSELoss()  # loss function defined;

optimizer = optim.Adam(model.parameters(), learning_rate) # gradient descent method based on average and squares of gradient

Train your model. Note how the defined batches need some adjustments in the code (evaluating loss...)

In [ ]:
# batched training
for epoch in range(num_epochs):

    model.train()
    total_loss = 0
    total_samples = 0

    for data, targets in train_loader:
        data = data.to(device)
        targets = targets.to(device)

        outputs = model(data.to(torch.float32))

        loss = criterion(outputs, targets)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        # accumulate loss over batches
        batch_size = data.size(0)
        total_loss += loss.item() * batch_size
        total_samples += batch_size

    avg_loss = total_loss / total_samples

    if (epoch + 1) % 10 == 0:
        print(f'Epoch [{epoch + 1}/{num_epochs}], Loss: {avg_loss:.4f}')

Evaluate the model using the test set and see if the NN shows over or underfitting. Adjust the model accordingly or train again.

In [ ]:
# Evaluate the model
model.eval()  # set model to eval mode (e.g. no dropout)

all_preds = []
all_targets = []

total_samples = 0
total_loss = 0

criterion = nn.MSELoss()

with torch.no_grad():
    for data, targets in test_loader:
        data = data.to(device)
        targets = targets.to(device)

        outputs = model(data.to(torch.float32))
        loss = criterion(outputs, targets)

        # accumulate loss over batches
        batch_size = data.size(0)
        total_loss += loss.item() * batch_size # multiply by batch size to 
        total_samples += targets.size(0)
        
        # Predictions      
        probs = torch.sigmoid(outputs)     # convert for metrics
        preds = (probs > 0.5).long()

        all_preds.extend(preds.cpu().tolist()) # cpu() moves the tensor to cpu (if on gpu), tolist converts it to python list
        all_targets.extend(targets.cpu().tolist())

avg_loss = total_loss / total_samples

print(f"Average test Loss: {avg_loss:.4f}")
